In [6]:
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

set_config(transform_output="pandas")

url = "https://drive.google.com/uc?export=download&id=1oYQSNxfvw6kFr6-N9rKLRAnLXlp0osEt"
songs_df = pd.read_csv(url)
songs_df.columns = songs_df.columns.str.strip()
songs_df = songs_df.drop(columns=['Unnamed: 0','mode','duration_ms','time_signature','id', 'html', 'key', 'type'])

songs_df = songs_df.set_index(["name", "artist"])

# Pick clustering dimensions from the continuous audio features.
# Popularity is excluded by default because it measures success, not sound.
numeric_df = songs_df.select_dtypes(include="number").copy()
preferred_dims = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence",
]

cluster_dims = [
    col
    for col in preferred_dims
    if col in numeric_df.columns and numeric_df[col].nunique(dropna=True) > 1
]

if not cluster_dims:
    cluster_dims = [
        col
        for col in numeric_df.columns
        if col != "popularity" and numeric_df[col].nunique(dropna=True) > 1
    ]

# Remove highly redundant features so the distance metric stays meaningful.
corr = numeric_df[cluster_dims].corr().abs() if len(cluster_dims) > 1 else None
selected_dims = []
for col in cluster_dims:
    if corr is None or all(corr.loc[col, kept] < 0.90 for kept in selected_dims):
        selected_dims.append(col)
cluster_dims = selected_dims

print("Selected clustering dimensions:", cluster_dims)

if len(songs_df) < 15:
    raise ValueError("Need at least 15 songs to build 10-15 clusters.")

X = numeric_df[cluster_dims]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.95, random_state=42)
X_cluster = pca.fit_transform(X_scaled)
print(f"PCA components retained: {pca.n_components_} -> {pca.explained_variance_ratio_.sum():.1%} variance")

k_candidates = range(10, min(16, len(songs_df)))
silhouette_by_k = {}
for k in k_candidates:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_cluster)
    silhouette_by_k[k] = silhouette_score(X_cluster, labels)

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
print("Silhouette by k:", silhouette_by_k)
print("Best k:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
songs_df = songs_df.copy()
songs_df["cluster"] = kmeans.fit_predict(X_cluster)

cluster_features = pd.DataFrame(X_cluster, index=songs_df.index)
overall_profile = songs_df[cluster_dims].mean()
overall_std = songs_df[cluster_dims].std(ddof=0).replace(0, np.nan)

def infer_cluster_name(cluster_id):
    cluster_profile = songs_df.loc[songs_df["cluster"] == cluster_id, cluster_dims].mean()
    z_scores = ((cluster_profile - overall_profile) / overall_std).dropna()
    ranked = z_scores.sort_values(ascending=False)

    energy = ranked.get("energy", 0)
    danceability = ranked.get("danceability", 0)
    acousticness = ranked.get("acousticness", 0)
    instrumentalness = ranked.get("instrumentalness", 0)
    liveness = ranked.get("liveness", 0)
    speechiness = ranked.get("speechiness", 0)
    loudness = ranked.get("loudness", 0)
    tempo = ranked.get("tempo", 0)
    valence = ranked.get("valence", 0)

    if instrumentalness >= 0.8:
        return "Instrumental / Ambient"
    if speechiness >= 0.8:
        return "Speech / Vocal Heavy"
    if acousticness >= 0.7 and energy <= -0.1:
        return "Acoustic / Chill"
    if energy >= 0.8 and danceability >= 0.3 and valence >= 0.1:
        return "High-Energy Dance"
    if energy >= 0.7 and loudness >= 0.4 and valence <= 0.0:
        return "Loud / Intense"
    if liveness >= 0.7:
        return "Live / Raw"
    if tempo >= 0.8 and energy >= 0.2:
        return "Fast / Driving"
    if danceability >= 0.8 and valence >= 0.2:
        return "Groovy / Feel-Good"
    if valence >= 0.8 and energy >= 0.0:
        return "Bright / Uplifting"
    if acousticness >= 0.4 and valence >= 0.1:
        return "Warm / Organic"
    if valence <= -0.2 and energy <= 0.3:
        return "Moody / Reflective"
    if energy >= 0.4 and danceability >= 0.3:
        return "Mainstream / Pop"
    return "Balanced / Mixed"

cluster_names = {cluster_id: infer_cluster_name(cluster_id) for cluster_id in sorted(songs_df["cluster"].unique())}

def build_cluster_playlist(cluster_id, n_songs=10):
    cluster_mask = songs_df["cluster"] == cluster_id
    cluster_songs = songs_df.loc[cluster_mask].copy()
    cluster_vectors = cluster_features.loc[cluster_mask]
    cluster_center = pd.Series(kmeans.cluster_centers_[cluster_id], index=cluster_vectors.columns)
    cluster_songs["distance_to_cluster_center"] = np.linalg.norm(
        cluster_vectors - cluster_center, axis=1
    )
    return cluster_songs.sort_values("distance_to_cluster_center").head(n_songs)

def cluster_theme(cluster_id):
    profile = songs_df.loc[songs_df["cluster"] == cluster_id, cluster_dims].mean().sort_values(ascending=False)
    top_features = profile.head(3).index.str.replace("_", " ")
    return f"{cluster_names[cluster_id]}: " + ", ".join(top_features)

cluster_summary = []
for cluster_id in sorted(cluster_names):
    top_feature_names = songs_df.loc[songs_df["cluster"] == cluster_id, cluster_dims].mean().sort_values(ascending=False).head(3).index.str.replace("_", " ")
    cluster_summary.append({
        "cluster": cluster_id,
        "category": cluster_names[cluster_id],
        "size": int((songs_df["cluster"] == cluster_id).sum()),
        "top_features": ", ".join(top_feature_names),
    })

cluster_summary_df = pd.DataFrame(cluster_summary).sort_values(["size", "cluster"], ascending=[False, True])
print("Cluster sizes:")
print(cluster_summary_df[["cluster", "category", "size"]])

sample_cluster = cluster_summary_df.iloc[0]["cluster"]
print(cluster_theme(sample_cluster))
sample_playlist = build_cluster_playlist(sample_cluster, n_songs=8)
sample_playlist

Selected clustering dimensions: ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']
PCA components retained: 7 -> 96.3% variance
Silhouette by k: {10: 0.20483290835444015, 11: 0.20423680053672155, 12: 0.20216728260200445, 13: 0.20451621063956554, 14: 0.1987422259417092, 15: 0.19901209374395315}
Best k: 10
Cluster sizes:
   cluster                category  size
4        4      Groovy / Feel-Good  1065
2        2        Balanced / Mixed   807
9        9  Instrumental / Ambient   743
8        8        Acoustic / Chill   581
7        7          Loud / Intense   425
1        1          Fast / Driving   403
3        3  Instrumental / Ambient   355
5        5    Speech / Vocal Heavy   306
6        6  Instrumental / Ambient   298
0        0              Live / Raw   252
Groovy / Feel-Good: tempo, valence, danceability


,,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,cluster,distance_to_cluster_center
name,artist,,,,,,,,,,,
Ain't No Stoppin' Us Now - Single Version,McFadden & Whitehead,0.721,0.743,-6.358,0.0525,0.0835,0.000000,0.1350,0.867,112.504,4,0.480708
Rockabye (feat. Sean Paul & Anne-Marie),Clean Bandit,0.720,0.763,-4.068,0.0523,0.4060,0.000000,0.1800,0.742,101.965,4,0.480736
Hecha Pa' Mi,Boza,0.725,0.756,-5.013,0.0572,0.3620,0.000685,0.1030,0.828,100.070,4,0.520750
Sexy Movimiento,Wisin & Yandel,0.815,0.757,-6.160,0.0390,0.3820,0.000003,0.1470,0.734,104.986,4,0.565799
Broadway Jungle - 2000 Version,Toots & The Maytals,0.771,0.775,-6.454,0.0405,0.2410,0.005910,0.1040,0.824,117.959,4,0.572317
Say You'll Be There,Spice Girls,0.726,0.679,-6.219,0.0473,0.0149,0.000001,0.0837,0.751,107.020,4,0.581249
Have You Ever Seen The Rain,Creedence Clearwater Revival,0.741,0.697,-7.028,0.0277,0.0664,0.000023,0.1330,0.774,116.109,4,0.582692
Fantasy,Mariah Carey,0.669,0.727,-7.588,0.0353,0.1390,0.000000,0.1230,0.807,102.322,4,0.602960


### Should you use another clustering approach?

For this dataset, the best default is to cluster on the continuous audio features after standardizing them. PCA is useful here because it removes redundancy and makes the distances cleaner, but it is not mandatory for clustering.

If you want a more practical playlist prototype, compare `AgglomerativeClustering` with `KMeans` on both the scaled features and the PCA-reduced features. `KMeans` is usually easier to turn into playlists because every song belongs to exactly one cluster and the clusters tend to be more compact.

Use `DBSCAN` only if you want to discover unusual dense pockets and outliers. It is less predictable for playlist building because it can label many songs as noise when the density is uneven.

The next cell compares clustering quality on the standardized space versus the PCA space so you can see whether decomposition is actually helping here.

In [3]:
from sklearn.cluster import KMeans

comparison_rows = []
comparison_spaces = {
    "scaled": X_scaled,
    "pca_95": X_cluster,
}

for space_name, space_data in comparison_spaces.items():
    agglomerative_labels = AgglomerativeClustering(n_clusters=best_k).fit_predict(space_data)
    kmeans_labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(space_data)

    comparison_rows.append({
        "space": space_name,
        "model": "Agglomerative",
        "silhouette": silhouette_score(space_data, agglomerative_labels),
    })
    comparison_rows.append({
        "space": space_name,
        "model": "KMeans",
        "silhouette": silhouette_score(space_data, kmeans_labels),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(["space", "silhouette"], ascending=[True, False])
comparison_df

,space,model,silhouette
3,pca_95,KMeans,0.341980
2,pca_95,Agglomerative,0.305522
1,scaled,KMeans,0.333466
0,scaled,Agglomerative,0.290740
